In [3]:
import pandas as pd
import argparse
import torch as tc
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, random_split
from PIL import Image
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Copy data from drive to local storage

!cp -r /content/drive/MyDrive/data /content/
print("Data copied to local storage /content/data")

Data copied to local storage /content/data


In [15]:
# Configuration

class Args():
    batch_size = 64
    test_batch_size = 128
    epochs = 14
    lr = 0.01 # Smaller learning rate
    gamma = 0.7
    dry_run = False
    seed = 1
    log_interval = 10
    save_model = True


In [6]:
# Image loading

DIR = Path("/content/data")

TRANSFORM_DEFAULT = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize((224, 224)),
        v2.ToDtype(tc.float32, scale=True),
    ]
)

class ImageClassification(Dataset):

    def __init__(
        self,
        csv_file: Path,
        transform: v2.Transform = TRANSFORM_DEFAULT,
    ):
        self.df = pd.read_csv(csv_file, header=0)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        label = int(row.label)
        path = row.image_path

        full_path = DIR / path.lstrip('/')

        image = Image.open(full_path).convert("RGB")
        image = self.transform(image)
        return idx, image, label, path

In [7]:
# Model generation

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1) # 3 input channels, as input is RGB
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(774400, 128) # Input features (t.b.d.)
        self.fc2 = nn.Linear(128, 201) # 200 output classes

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = tc.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [8]:
# Training

def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (_, data, target, _) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print("Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}"
                .format(
                    epoch,
                    batch_idx * len(data),
                    len(train_loader.dataset),
                    100.0 * batch_idx / len(train_loader),
                    loss.item()))
            if args.dry_run:
                break

In [9]:
# Testing

def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with tc.no_grad():
        for _, data, target, _ in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            # sum up batch loss
            test_loss += F.nll_loss(output, target, reduction="sum").item()

            # get the index of the max log-probability
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print("\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n"
        .format(
            test_loss,
            correct,
            len(test_loader.dataset),
            100.0 * correct / len(test_loader.dataset)))

In [16]:
# Full model training

args = Args()

tc.manual_seed(args.seed)

if tc.cuda.is_available():
    device = tc.device("cuda")
    print("Using GPU")
else:
    device = tc.device("cpu")
    print("Using CPU")

IMAGE_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((NORM_MEAN), (NORM_STD))
    ]
)

dataset = ImageClassification(
    csv_file = DIR / "train_images.csv",
    transform = transform
)

# Split into training and test set
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_subset, test_subset = random_split(dataset, [train_size, test_size])

train_loader = tc.utils.data.DataLoader(
    train_subset,
    batch_size=args.batch_size,
    shuffle=True
)
test_loader = tc.utils.data.DataLoader(
    test_subset,
    batch_size=args.test_batch_size
)

model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=args.lr)

scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
for epoch in range(1, args.epochs + 1):
    train(args, model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

if args.save_model:
    tc.save(model.state_dict(), "feathersinfocus_cnn.pt")

Using GPU
Train Epoch: 1 [0/3140 (0%)]	Loss: 5.309593
Train Epoch: 1 [640/3140 (20%)]	Loss: 5.274199
Train Epoch: 1 [1280/3140 (40%)]	Loss: 5.362281
Train Epoch: 1 [1920/3140 (60%)]	Loss: 5.292458
Train Epoch: 1 [2560/3140 (80%)]	Loss: 5.320278

Test set: Average loss: 5.3028, Accuracy: 5/786 (1%)

Train Epoch: 2 [0/3140 (0%)]	Loss: 5.296347
Train Epoch: 2 [640/3140 (20%)]	Loss: 5.309320
Train Epoch: 2 [1280/3140 (40%)]	Loss: 5.299570
Train Epoch: 2 [1920/3140 (60%)]	Loss: 5.288111
Train Epoch: 2 [2560/3140 (80%)]	Loss: 5.296966

Test set: Average loss: 5.2983, Accuracy: 4/786 (1%)

Train Epoch: 3 [0/3140 (0%)]	Loss: 5.296560
Train Epoch: 3 [640/3140 (20%)]	Loss: 5.303959
Train Epoch: 3 [1280/3140 (40%)]	Loss: 5.307343
Train Epoch: 3 [1920/3140 (60%)]	Loss: 5.307774
Train Epoch: 3 [2560/3140 (80%)]	Loss: 5.297022

Test set: Average loss: 5.3065, Accuracy: 6/786 (1%)

Train Epoch: 4 [0/3140 (0%)]	Loss: 5.269649
Train Epoch: 4 [640/3140 (20%)]	Loss: 5.303421
Train Epoch: 4 [1280/3140 (40

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
# Empty cache to avoid OutOfMemoryError

tc.cuda.empty_cache()